# Chapter 01: Introduction

Source orientation: *Fundamentals of Computer Graphics*, Chapter 01, printed pages 1-12, physical PDF pages 18-29. The source span introduces the field, names the major work areas and applications, frames graphics APIs as contracts between programs and devices, sketches the triangle-oriented graphics pipeline, and then gives practical advice about floating point, efficiency, class design, precision, and visual debugging.

## Chapter goal

The goal of this notebook is to turn the introductory vocabulary into inspectable computational objects. A reader should leave with a working mental model of how graphics programs move from descriptions of a scene to pixels, why APIs and pipelines divide responsibility the way they do, and why numerical and debugging habits matter as much as drawing a pretty image.

This chapter is not a historical survey in notebook form. It is a small lab bench for the ideas that recur throughout the rest of the course. We will build tiny versions of an application map, an API/pipeline boundary diagram, a raster route, a ray route, floating-point edge cases, a memory-locality/level-of-detail tradeoff, and visual debugging images. Every artifact is generated from code in this notebook and is saved under `artifacts/chapter-01`.

## Translation guide

| Chapter phrase | Computational object in this notebook | Inspection target |
| --- | --- | --- |
| Modeling | Vertices, object IDs, normals, and small scene data structures | What must exist before an image can be made |
| Rendering | A procedure that turns scene state into pixel values | Which inputs affect light, visibility, and color |
| Animation | Time-indexed changes to model or camera state | Why motion is an added dimension, not a separate renderer |
| Graphics API | A boundary of named operations and data promises | Which responsibilities belong to application code, UI code, and drawing code |
| Graphics pipeline | A sequence of data transformations from object space to a framebuffer | Where geometry becomes screen samples and where depth resolves visibility |
| Homogeneous coordinates | 4-vectors and 4 x 4 matrices followed by a divide by `w` | Why perspective is easier to compose before the divide |
| IEEE floating point | `inf`, `-inf`, `nan`, and signed zero behavior | Which branch conditions are well-defined and which values need traps |
| Efficiency | Coherent memory access and adjustable geometric detail | Why operation counts alone do not predict performance |
| Visual debugging | Images that encode internal program state | How to turn a conceptual bug into something observable |

The library choices are intentionally small. NumPy carries array math and floating-point experiments. Matplotlib creates durable labeled PNGs for static diagrams and debug images. NetworkX exposes the relationship structure among graphics areas and applications. Plotly is used where an interactive camera/projection artifact makes the 3D-to-2D pipeline easier to inspect. Pillow-backed helpers validate that image artifacts are nonblank. These choices match the chapter's architecture and debugging focus; heavier mesh, topology, manifold, or computer-vision libraries would be premature here.


In [ ]:
from pathlib import Path
import math
import sys

# Find the FCG book root whether the notebook is executed from the chapter
# directory or from the workspace root.
search_starts = [Path.cwd(), Path.cwd() / "Fundamentals-of-Computer-Graphics"]
BOOK_ROOT = None
for start in search_starts:
    for candidate in [start, *start.parents]:
        if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
            BOOK_ROOT = candidate
            break
    if BOOK_ROOT is not None:
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

CHAPTER = 1
TITLE = "Introduction"
TOPIC = f"chapter-{CHAPTER:02d}"
PRINTED_PAGES = "1-12"
PDF_PAGES = "18-29"
ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for kind in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / kind).mkdir(parents=True, exist_ok=True)

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Polygon
import networkx as nx
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    save_json,
    save_matplotlib,
    save_plotly_html,
    save_table_csv,
)
from utils.graphics_math import barycentric_2d, normalize
from utils.notebook_checks import assert_nonblank_image
from utils.plotting import PALETTE, close
from utils.transforms import homogeneous_divide, look_at, perspective, viewport

np.set_printoptions(precision=4, suppress=True)
ARTIFACTS = {"figures": [], "html": [], "checks": [], "tables": [], "data": []}
CHECKS = {"chapter": CHAPTER, "printed_pages": PRINTED_PAGES, "pdf_pages": PDF_PAGES}


def remember(path, kind):
    path = Path(path)
    ARTIFACTS[kind].append(path)
    return path


def fig_caption(path, width=760):
    display_artifact(path, width=width)
    return book_relative(path)


## Visual storyboard

The planner pass for this chapter routes each source topic to an inspectable artifact rather than to a decorative illustration.

| Source-grounded topic | Representation | Artifact | Validation target |
| --- | --- | --- | --- |
| Graphics areas and applications | Bipartite dependency map with an application routing table | `graphics-areas-application-map.png`, `graphics-application-routing.csv` | Every application connects to more than one graphics area |
| API and pipeline boundaries | Responsibility diagram plus stage-by-stage data flow | `api-pipeline-responsibility-map.png` | The drawing route has ordered model, transform, clip, raster, shade, display stages |
| Modeling/rendering/raster/ray overview | Side-by-side miniature rasterizer and ray caster | `raster-ray-rendering-routes.png`, `pipeline-stage-data.csv` | Z-buffer keeps the nearer sample and ray hits are finite where geometry is visible |
| 4D homogeneous pipeline | Interactive Plotly object/camera/screen inspector | `homogeneous-pipeline-inspector.html` | Clip `w` is positive and homogeneous divide yields finite normalized coordinates |
| IEEE floating-point behavior | Sentinel table and branch-condition plot | `ieee-floating-point-sentinels.png` | `nan > 0` is false, divide-by-zero yields infinities, signed zero is detectable |
| Efficiency | Cache-line traversal and level-of-detail tradeoff | `efficiency-locality-lod-budget.png` | Row-major traversal has fewer cache-line jumps than column-major traversal |
| Debugging graphics programs | Normal, object-ID, work, and error-mask debug images | `visual-debugging-channels.png`, `debugging-hypothesis-tests.csv` | The bug hypothesis reduces the number of invalid pixels after a local fix |

This storyboard deliberately uses Plotly only for the camera/perspective artifact, where interaction helps. The other visuals are labeled static PNGs because the learning target is comparison, validation, and durable inspection.


In [ ]:
storyboard = [
    {
        "concept": "graphics areas and applications",
        "representation": "bipartite dependency map plus routing table",
        "library": "NetworkX + Matplotlib",
        "artifact": "figures/graphics-areas-application-map.png",
        "inspection_target": "applications depend on combinations of modeling, rendering, animation, images, interaction, and visualization",
        "validation": "minimum application degree is greater than one",
    },
    {
        "concept": "API and pipeline boundaries",
        "representation": "responsibility diagram and ordered data-flow stages",
        "library": "Matplotlib",
        "artifact": "figures/api-pipeline-responsibility-map.png",
        "inspection_target": "application, UI API, graphics API, driver/GPU, and framebuffer own different promises",
        "validation": "pipeline stage list is ordered and complete",
    },
    {
        "concept": "raster and ray rendering routes",
        "representation": "mini z-buffer rasterizer and mini ray caster",
        "library": "NumPy + Matplotlib",
        "artifact": "figures/raster-ray-rendering-routes.png",
        "inspection_target": "raster visibility is per sample; ray visibility is nearest positive intersection",
        "validation": "z-buffer chooses the nearer triangle and ray hit distances are finite",
    },
    {
        "concept": "homogeneous graphics pipeline",
        "representation": "interactive 3D-to-2D inspector",
        "library": "Plotly",
        "artifact": "html/homogeneous-pipeline-inspector.html",
        "inspection_target": "object vertices become clip coordinates, then normalized device coordinates after division by w",
        "validation": "clip w positive and NDC coordinates finite",
    },
    {
        "concept": "IEEE floating-point sentinels",
        "representation": "numeric table and branch-condition diagram",
        "library": "NumPy + Matplotlib",
        "artifact": "figures/ieee-floating-point-sentinels.png",
        "inspection_target": "inf, -inf, nan, and signed zero can simplify or break graphics code paths",
        "validation": "nan comparison false and signed zero signbit true",
    },
    {
        "concept": "efficiency through locality and LOD",
        "representation": "cache-line traversal sketch and triangle-budget curve",
        "library": "NumPy + Matplotlib",
        "artifact": "figures/efficiency-locality-lod-budget.png",
        "inspection_target": "memory access pattern and detail level dominate many graphics costs",
        "validation": "coherent traversal has fewer cache-line switches",
    },
    {
        "concept": "visual debugging",
        "representation": "debug render targets and hypothesis table",
        "library": "NumPy + Matplotlib + Pillow helpers",
        "artifact": "figures/visual-debugging-channels.png",
        "inspection_target": "internal state can be encoded directly into image channels",
        "validation": "the fix reduces invalid pixels in the suspicious region",
    },
]
storyboard_path = remember(save_json(storyboard, TOPIC, "visual-storyboard.json"), "checks")
display_artifact(storyboard_path)


## Graphics areas and applications

The chapter separates core work areas from application domains. The important point is that applications do not use just one area. A film shot may combine modeling, rendering, animation, image processing, and compositing. A CAD/CAM workflow may care more about exact geometry and manufacturing constraints than about photorealistic lighting. Medical imaging and information visualization may start with measured data rather than with synthetic objects.

The graph below makes that dependency structure visible. Blue nodes are graphics work areas. Gold nodes are application domains. The edges are not a taxonomy claim; they are a practical routing map for deciding which later chapters a project will draw on.


In [ ]:
areas = [
    "modeling",
    "rendering",
    "animation",
    "interaction",
    "visualization",
    "image processing",
    "3D scanning",
    "computational photography",
]
applications = [
    "video games",
    "cartoons",
    "visual effects",
    "animated films",
    "CAD/CAM",
    "simulation",
    "medical imaging",
    "information visualization",
]
edges = [
    ("video games", "modeling"), ("video games", "rendering"), ("video games", "animation"), ("video games", "interaction"),
    ("cartoons", "modeling"), ("cartoons", "rendering"), ("cartoons", "animation"),
    ("visual effects", "modeling"), ("visual effects", "rendering"), ("visual effects", "animation"), ("visual effects", "image processing"), ("visual effects", "computational photography"),
    ("animated films", "modeling"), ("animated films", "rendering"), ("animated films", "animation"),
    ("CAD/CAM", "modeling"), ("CAD/CAM", "3D scanning"), ("CAD/CAM", "visualization"),
    ("simulation", "modeling"), ("simulation", "rendering"), ("simulation", "interaction"), ("simulation", "animation"),
    ("medical imaging", "visualization"), ("medical imaging", "image processing"), ("medical imaging", "3D scanning"),
    ("information visualization", "visualization"), ("information visualization", "interaction"), ("information visualization", "image processing"),
]

G = nx.Graph()
G.add_nodes_from(areas, kind="area")
G.add_nodes_from(applications, kind="application")
G.add_edges_from(edges)

area_y = np.linspace(1.0, 0.0, len(areas))
app_y = np.linspace(1.0, 0.0, len(applications))
pos = {name: (0.0, y) for name, y in zip(areas, area_y)}
pos.update({name: (1.0, y) for name, y in zip(applications, app_y)})

fig, ax = plt.subplots(figsize=(10.5, 6.2))
ax.set_facecolor(PALETTE["paper"])
for a, b in G.edges:
    ax.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]], color="#aeb8c5", lw=1.25, alpha=0.72, zorder=1)
for node in areas:
    x, y = pos[node]
    ax.scatter(x, y, s=900, color=PALETTE["blue"], edgecolor="white", linewidth=1.5, zorder=3)
    ax.text(x - 0.04, y, node, ha="right", va="center", fontsize=9, color=PALETTE["ink"])
for node in applications:
    x, y = pos[node]
    ax.scatter(x, y, s=900, color=PALETTE["gold"], edgecolor="white", linewidth=1.5, zorder=3)
    ax.text(x + 0.04, y, node, ha="left", va="center", fontsize=9, color=PALETTE["ink"])
ax.text(0.0, 1.08, "work areas", ha="center", va="bottom", fontsize=11, weight="bold", color=PALETTE["ink"])
ax.text(1.0, 1.08, "application domains", ha="center", va="bottom", fontsize=11, weight="bold", color=PALETTE["ink"])
ax.text(0.5, -0.08, "A project usually crosses several areas; that is why graphics code needs shared geometry, image, and debugging habits.", ha="center", fontsize=9, color=PALETTE["gray"])
ax.set_xlim(-0.45, 1.45)
ax.set_ylim(-0.14, 1.14)
ax.axis("off")
map_path = remember(save_matplotlib(fig, TOPIC, "graphics-areas-application-map.png"), "figures")
close(fig)

routing_rows = []
for app in applications:
    neighbors = sorted(G.neighbors(app))
    routing_rows.append({
        "application": app,
        "connected_areas": "; ".join(neighbors),
        "area_count": len(neighbors),
        "first_debug_question": "Which intermediate state can be made visible?",
    })
routing_table_path = remember(save_table_csv(routing_rows, TOPIC, "graphics-application-routing.csv"), "tables")

app_degrees = {app: G.degree(app) for app in applications}
CHECKS["application_graph"] = {
    "area_count": len(areas),
    "application_count": len(applications),
    "edge_count": G.number_of_edges(),
    "minimum_application_degree": int(min(app_degrees.values())),
    "connected": bool(nx.is_connected(G)),
}
assert CHECKS["application_graph"]["minimum_application_degree"] > 1
fig_caption(map_path)
display_artifact(routing_table_path)


## APIs and pipeline responsibility

A graphics API is useful because it lets application code ask for visual output without spelling out every driver and device detail. The same program also needs a user-interface API for input. In practice, the application owns scene data and intent; UI code owns events; the graphics API owns draw commands and resource handles; the driver and hardware own a fast implementation of the pipeline; the framebuffer is the observable image.

The diagram tracks that boundary. The lower row names the chapter's pipeline stages as data transformations. The key inspection target is that an API call is not magic: it transfers structured data into a pipeline whose later stages can be debugged as arrays, coordinates, masks, and images.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))
ax.set_facecolor(PALETTE["paper"])
ax.axis("off")

boxes = [
    ("application code", "scene objects\nmaterials\nprogram intent", 0.04, 0.62, 0.18, 0.22, PALETTE["blue"]),
    ("user-interface API", "mouse/tablet\nkeyboard\nwindow events", 0.29, 0.72, 0.18, 0.16, PALETTE["teal"]),
    ("graphics API", "buffers\ntextures\nstate\ndraw calls", 0.29, 0.48, 0.18, 0.18, PALETTE["gold"]),
    ("driver + GPU", "validated commands\nparallel execution\npipeline hardware", 0.54, 0.58, 0.19, 0.22, PALETTE["violet"]),
    ("framebuffer", "pixels\ndepth\ndebug targets", 0.80, 0.62, 0.16, 0.18, PALETTE["green"]),
]
for title, body, x, y, w, h, color in boxes:
    ax.add_patch(Rectangle((x, y), w, h, facecolor=color, edgecolor="white", linewidth=1.5, alpha=0.92))
    ax.text(x + w / 2, y + h - 0.035, title, ha="center", va="top", fontsize=10, color="white", weight="bold")
    ax.text(x + w / 2, y + 0.045, body, ha="center", va="bottom", fontsize=8.5, color="white")

arrows = [
    ((0.22, 0.74), (0.29, 0.80), "events"),
    ((0.22, 0.66), (0.29, 0.57), "draw intent"),
    ((0.47, 0.57), (0.54, 0.66), "commands"),
    ((0.73, 0.68), (0.80, 0.70), "samples"),
    ((0.88, 0.62), (0.13, 0.62), "observe/debug", "#c44e52"),
]
for item in arrows:
    start, end, label = item[:3]
    color = item[3] if len(item) == 4 else PALETTE["ink"]
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=13, lw=1.4, color=color, connectionstyle="arc3,rad=0.08"))
    lx = (start[0] + end[0]) / 2
    ly = (start[1] + end[1]) / 2
    ax.text(lx, ly + 0.035, label, ha="center", fontsize=8, color=color)

pipeline_stages = [
    ("model", "vertices\nattributes"),
    ("transform", "4 x 4 matrices"),
    ("clip", "homogeneous half-spaces"),
    ("rasterize", "covered samples"),
    ("shade", "colors + depth"),
    ("display", "image"),
]
x0, y0, w, h, gap = 0.05, 0.18, 0.135, 0.14, 0.027
for i, (stage, data) in enumerate(pipeline_stages):
    x = x0 + i * (w + gap)
    ax.add_patch(Rectangle((x, y0), w, h, facecolor="#ffffff", edgecolor="#9aa8b8", linewidth=1.2))
    ax.text(x + w / 2, y0 + h - 0.035, stage, ha="center", va="top", fontsize=9, weight="bold", color=PALETTE["ink"])
    ax.text(x + w / 2, y0 + 0.035, data, ha="center", va="bottom", fontsize=7.8, color=PALETTE["gray"])
    if i < len(pipeline_stages) - 1:
        ax.add_patch(FancyArrowPatch((x + w, y0 + h / 2), (x + w + gap, y0 + h / 2), arrowstyle="-|>", mutation_scale=12, lw=1.2, color=PALETTE["ink"]))
ax.text(0.5, 0.095, "Pipeline thinking: each stage changes the representation, so each stage can expose a different debugging view.", ha="center", fontsize=9, color=PALETTE["ink"])

api_path = remember(save_matplotlib(fig, TOPIC, "api-pipeline-responsibility-map.png"), "figures")
close(fig)
CHECKS["pipeline_responsibility"] = {
    "api_boundary_count": len(boxes),
    "pipeline_stage_count": len(pipeline_stages),
    "ordered_stages": [stage for stage, _ in pipeline_stages],
}
assert CHECKS["pipeline_responsibility"]["ordered_stages"] == ["model", "transform", "clip", "rasterize", "shade", "display"]
fig_caption(api_path)


## Modeling, rendering, rasterization, and ray tracing

The chapter's pipeline discussion focuses on efficient triangle drawing, but it also prepares us for a broader question: how does a modeled scene become pixel values? The two miniature renderers below answer that question in different ways.

The raster route starts with screen-space triangles. It tests which samples fall inside each triangle, interpolates depth, and keeps the closest sample in a z-buffer. The ray route starts at the camera and asks which object a ray hits first. Both are visibility algorithms. They differ in traversal order: rasterization loops over primitives and samples; ray tracing loops over rays and intersections.


In [ ]:
def barycentric_grid(width, height, tri_xy):
    y, x = np.mgrid[0:height, 0:width]
    p = np.stack([x + 0.5, y + 0.5], axis=-1)
    a, b, c = [np.asarray(v, dtype=float) for v in tri_xy]
    denom = (b[1] - c[1]) * (a[0] - c[0]) + (c[0] - b[0]) * (a[1] - c[1])
    w0 = ((b[1] - c[1]) * (p[..., 0] - c[0]) + (c[0] - b[0]) * (p[..., 1] - c[1])) / denom
    w1 = ((c[1] - a[1]) * (p[..., 0] - c[0]) + (a[0] - c[0]) * (p[..., 1] - c[1])) / denom
    w2 = 1.0 - w0 - w1
    return np.stack([w0, w1, w2], axis=-1)

width = height = 190
background = np.ones((height, width, 3)) * np.array([0.965, 0.975, 0.985])
z_buffer = np.ones((height, width)) * np.inf
raster_rgb = background.copy()
owner = np.full((height, width), -1, dtype=int)
triangles = [
    {
        "xy": np.array([[35, 152], [91, 28], [164, 145]], dtype=float),
        "z": np.array([0.62, 0.46, 0.70]),
        "color": np.array([[0.20, 0.45, 0.78], [0.36, 0.70, 0.55], [0.95, 0.68, 0.18]]),
        "name": "blue near vertex",
    },
    {
        "xy": np.array([[54, 120], [142, 58], [151, 171]], dtype=float),
        "z": np.array([0.38, 0.82, 0.50]),
        "color": np.array([[0.78, 0.26, 0.30], [0.52, 0.36, 0.76], [0.95, 0.84, 0.32]]),
        "name": "red near edge",
    },
]
for tri_index, tri in enumerate(triangles):
    bary = barycentric_grid(width, height, tri["xy"])
    inside = np.all(bary >= -1e-9, axis=-1)
    depth = np.tensordot(bary, tri["z"], axes=([2], [0]))
    update = inside & (depth < z_buffer)
    color = np.tensordot(np.clip(bary, 0, 1), tri["color"], axes=([2], [0]))
    raster_rgb[update] = color[update]
    z_buffer[update] = depth[update]
    owner[update] = tri_index

ys, xs = np.mgrid[-1.0:1.0:complex(height), -1.0:1.0:complex(width)]
origins = np.dstack([np.zeros_like(xs), np.zeros_like(ys), np.ones_like(xs) * 2.8])
dirs = np.dstack([xs, ys, -np.ones_like(xs) * 2.2])
dirs = dirs / np.linalg.norm(dirs, axis=-1, keepdims=True)
center = np.array([0.0, 0.0, 0.0])
radius = 0.82
oc = origins - center
a = np.sum(dirs * dirs, axis=-1)
b = 2.0 * np.sum(oc * dirs, axis=-1)
c = np.sum(oc * oc, axis=-1) - radius * radius
disc = b * b - 4.0 * a * c
ray_rgb = background.copy()
ray_t = np.ones((height, width)) * np.inf
hit = disc >= 0.0
sqrt_disc = np.sqrt(np.clip(disc, 0.0, None))
t0 = (-b - sqrt_disc) / (2.0 * a)
valid = hit & (t0 > 0.0)
ray_t[valid] = t0[valid]
points = origins + dirs * ray_t[..., None]
normals = np.zeros_like(points)
normals[valid] = (points[valid] - center) / radius
light = normalize(np.array([-0.4, 0.65, 1.0]))
shade = np.clip(np.sum(normals * light, axis=-1), 0.0, 1.0)
base = np.array([0.31, 0.58, 0.86])
ray_rgb[valid] = 0.08 + shade[valid, None] * base

visible_depth = np.where(np.isfinite(z_buffer), z_buffer, np.nan)
fig, axes = plt.subplots(1, 3, figsize=(11, 4.1))
axes[0].imshow(raster_rgb)
for tri in triangles:
    axes[0].add_patch(Polygon(tri["xy"], closed=True, fill=False, edgecolor="white", linewidth=1.2))
axes[0].set_title("raster route: triangles -> samples")
axes[0].axis("off")

im = axes[1].imshow(visible_depth, cmap="viridis_r")
axes[1].set_title("z-buffer: nearest stored depth")
axes[1].axis("off")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.02, label="depth")

axes[2].imshow(ray_rgb)
axes[2].set_title("ray route: nearest positive hit")
axes[2].axis("off")
fig.suptitle("Two visibility routes from scene description to pixels", fontsize=12, color=PALETTE["ink"])
render_path = remember(save_matplotlib(fig, TOPIC, "raster-ray-rendering-routes.png"), "figures")
close(fig)

pipeline_rows = [
    {"stage": "model", "data": "object vertices, object IDs, normals, material parameters", "debug_view": "wireframe, ID colors, normal colors"},
    {"stage": "transform", "data": "4-vectors multiplied by model/view/projection matrices", "debug_view": "basis axes and clip-coordinate ranges"},
    {"stage": "clip", "data": "homogeneous inequalities before perspective divide", "debug_view": "which vertices or primitives leave the view volume"},
    {"stage": "rasterize", "data": "sample coverage and barycentric coordinates", "debug_view": "coverage masks and barycentric color ramps"},
    {"stage": "depth", "data": "per-sample nearest depth", "debug_view": "z-buffer heat map"},
    {"stage": "shade", "data": "normals, light directions, textures, and material state", "debug_view": "normal maps, light terms, out-of-range masks"},
]
stage_table_path = remember(save_table_csv(pipeline_rows, TOPIC, "pipeline-stage-data.csv"), "tables")

sample = (95, 100)
winning_owner = int(owner[sample[1], sample[0]])
covering_depths = []
for tri in triangles:
    bary = barycentric_2d(np.array(sample, dtype=float) + 0.5, *tri["xy"])
    if np.all(bary >= -1e-9):
        covering_depths.append(float(bary @ tri["z"]))
z_winner_is_min = bool(covering_depths and math.isclose(float(z_buffer[sample[1], sample[0]]), min(covering_depths), rel_tol=0, abs_tol=1e-12))
CHECKS["rendering_routes"] = {
    "raster_covered_pixels": int(np.isfinite(z_buffer).sum()),
    "z_buffer_sample_owner": winning_owner,
    "z_buffer_closer_wins": z_winner_is_min,
    "ray_hit_pixels": int(valid.sum()),
    "nearest_ray_t": float(np.nanmin(ray_t[np.isfinite(ray_t)])),
}
assert CHECKS["rendering_routes"]["z_buffer_closer_wins"]
assert CHECKS["rendering_routes"]["ray_hit_pixels"] > 0
fig_caption(render_path)
display_artifact(stage_table_path)


## Homogeneous coordinates as the pipeline's bookkeeping

The chapter points ahead to one of the hardest early ideas in graphics: 3D points are often carried through a 4D homogeneous coordinate pipeline before the final perspective divide. The reason is composability. Model, view, and projection transformations can be multiplied as matrices, and the division by `w` is delayed until after clipping and projection.

Open the HTML artifact below and hover over the triangle vertices. The left pane shows the camera and a small object-space triangle. The right pane shows the same vertices after view/projection and homogeneous divide. The values to inspect are clip `w` and normalized device coordinates; if `w` is wrong, the final 2D position will be wrong even when the earlier 3D coordinates looked plausible.


In [ ]:
object_vertices = np.array([
    [-0.85, -0.55, 0.00],
    [0.85, -0.42, 0.15],
    [-0.10, 0.78, -0.10],
], dtype=float)
object_loop = np.vstack([object_vertices, object_vertices[0]])
eye = np.array([2.4, 1.7, 3.4])
target = np.array([0.0, 0.0, 0.0])
view, camera_basis = look_at(eye, target, np.array([0.0, 1.0, 0.0]))
proj = perspective(48.0, 4 / 3, 0.5, 20.0)
homo = np.column_stack([object_vertices, np.ones(len(object_vertices))])
clip = (proj @ view @ homo.T).T
ndc = homogeneous_divide(clip)
screen = viewport(ndc[:, :2], 640, 480)

hover_lines = []
for i, (v, c, n, s) in enumerate(zip(object_vertices, clip, ndc, screen)):
    hover_lines.append(
        f"vertex {i}<br>object=({v[0]:.2f}, {v[1]:.2f}, {v[2]:.2f})"
        f"<br>clip=({c[0]:.2f}, {c[1]:.2f}, {c[2]:.2f}, w={c[3]:.2f})"
        f"<br>ndc=({n[0]:.2f}, {n[1]:.2f}, {n[2]:.2f})"
        f"<br>screen=({s[0]:.1f}, {s[1]:.1f})"
    )

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "xy"}]],
    subplot_titles=("object, camera, and view rays", "after projection and divide by w"),
    horizontal_spacing=0.08,
)
fig.add_trace(
    go.Scatter3d(
        x=object_loop[:, 0], y=object_loop[:, 1], z=object_loop[:, 2],
        mode="lines+markers",
        marker=dict(size=5, color="#2f6fbb"),
        line=dict(color="#2f6fbb", width=5),
        name="model triangle",
        text=hover_lines + [hover_lines[0]],
        hovertemplate="%{text}<extra></extra>",
    ),
    row=1,
    col=1,
)
for v in object_vertices:
    ray = np.vstack([eye, v])
    fig.add_trace(
        go.Scatter3d(x=ray[:, 0], y=ray[:, 1], z=ray[:, 2], mode="lines", line=dict(color="#7a8793", width=2), showlegend=False),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter3d(x=[eye[0]], y=[eye[1]], z=[eye[2]], mode="markers+text", marker=dict(size=5, color="#c44e52"), text=["camera"], textposition="top center", name="camera"),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=np.r_[ndc[:, 0], ndc[0, 0]],
        y=np.r_[ndc[:, 1], ndc[0, 1]],
        mode="lines+markers",
        marker=dict(size=9, color="#d59f0f"),
        line=dict(color="#d59f0f", width=3),
        name="NDC triangle",
        text=hover_lines + [hover_lines[0]],
        hovertemplate="%{text}<extra></extra>",
    ),
    row=1,
    col=2,
)
fig.add_shape(type="rect", x0=-1, y0=-1, x1=1, y1=1, line=dict(color="#1f2933", dash="dash"), row=1, col=2)
fig.update_xaxes(range=[-1.2, 1.2], zeroline=True, title="NDC x", row=1, col=2)
fig.update_yaxes(range=[-1.2, 1.2], scaleanchor="x", scaleratio=1, zeroline=True, title="NDC y", row=1, col=2)
fig.update_layout(
    title="Homogeneous pipeline inspector",
    margin=dict(l=10, r=10, t=65, b=20),
    width=980,
    height=520,
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", aspectmode="data"),
)
html_path = remember(save_plotly_html(fig, TOPIC, "homogeneous-pipeline-inspector.html", include_plotlyjs=True), "html")
CHECKS["homogeneous_pipeline"] = {
    "clip_w_min": float(clip[:, 3].min()),
    "clip_w_all_positive": bool(np.all(clip[:, 3] > 0.0)),
    "ndc_all_finite": bool(np.isfinite(ndc).all()),
    "ndc_xy_inside_unit_square": bool(np.all(np.abs(ndc[:, :2]) <= 1.0)),
    "camera_basis_dot_uv": float(np.dot(camera_basis["u"], camera_basis["v"])),
}
assert CHECKS["homogeneous_pipeline"]["clip_w_all_positive"]
assert CHECKS["homogeneous_pipeline"]["ndc_all_finite"]
fig_caption(html_path, width="100%")


## Numerical issues: sentinels are part of the design

Graphics code is numerical code. The chapter's practical message is not merely that floating point has errors; it is that special values can be used deliberately when the surrounding logic expects them. Infinities can represent a ray with no current hit or a depth that has not been filled. NaN can poison an invalid computation so that a comparison fails instead of silently taking a branch. Signed zero is rare, but it matters in boundary cases where direction and side tests are encoded in signs.

The table below recreates a few of those cases with NumPy's IEEE-style behavior. The point is to decide which outcomes are useful sentinels and which should be trapped with assertions or debug colors.


In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    plus_inf = (np.array([2.0]) / np.array([0.0]))[0]
    minus_inf = (np.array([-2.0]) / np.array([0.0]))[0]
    nan_value = (np.array([0.0]) / np.array([0.0]))[0]
    signed_zero = (np.array([-2.0]) / np.array([np.inf]))[0]
    harmonic_parallel = (1.0 / (1.0 / np.array([0.0]) + 1.0 / np.array([4.0])))[0]

float_rows = [
    {"expression": "+a / +0", "result": str(plus_inf), "useful_graphics_reading": "unfilled depth or no finite bound"},
    {"expression": "-a / +0", "result": str(minus_inf), "useful_graphics_reading": "opposite unbounded direction"},
    {"expression": "0 / 0", "result": str(nan_value), "useful_graphics_reading": "invalid state that should not pass comparisons"},
    {"expression": "-a / +inf", "result": str(signed_zero), "useful_graphics_reading": "boundary value with a sign bit"},
    {"expression": "1 / (1/0 + 1/4)", "result": str(harmonic_parallel), "useful_graphics_reading": "zero contribution without a special branch"},
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [1.35, 1.0]})
axes[0].axis("off")
table = axes[0].table(
    cellText=[[row["expression"], row["result"], row["useful_graphics_reading"]] for row in float_rows],
    colLabels=["operation", "result", "graphics reading"],
    loc="center",
    cellLoc="left",
    colLoc="left",
    colWidths=[0.22, 0.16, 0.58],
)
table.auto_set_font_size(False)
table.set_fontsize(8.1)
table.scale(1, 1.5)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#d0d7de")
    if r == 0:
        cell.set_facecolor("#e8eef6")
        cell.set_text_props(weight="bold", color=PALETTE["ink"])
axes[0].set_title("IEEE-style sentinels used as program states", fontsize=11, color=PALETTE["ink"])

values = np.array([-np.inf, -1.0, signed_zero, 0.0, 1.0, np.inf])
labels = ["-inf", "-1", "-0", "+0", "+1", "+inf"]
xpos = np.arange(len(values))
colors = [PALETTE["red"], PALETTE["gray"], PALETTE["gold"], PALETTE["teal"], PALETTE["gray"], PALETTE["blue"]]
axes[1].scatter(xpos, np.zeros_like(xpos), s=130, c=colors, zorder=3)
for x, label in zip(xpos, labels):
    axes[1].text(x, 0.08, label, ha="center", fontsize=9)
axes[1].scatter([len(values) + 0.6], [0.0], s=150, marker="x", color=PALETTE["violet"], linewidth=3)
axes[1].text(len(values) + 0.6, 0.08, "NaN\ncomparison false", ha="center", fontsize=9)
axes[1].axhline(0, color="#b6c0ca", lw=1)
axes[1].set_ylim(-0.22, 0.32)
axes[1].set_yticks([])
axes[1].set_xticks([])
axes[1].set_title("Branches can treat sentinels deliberately")
for spine in axes[1].spines.values():
    spine.set_visible(False)

float_path = remember(save_matplotlib(fig, TOPIC, "ieee-floating-point-sentinels.png"), "figures")
close(fig)
CHECKS["floating_point"] = {
    "positive_divide_zero_is_pos_inf": bool(np.isposinf(plus_inf)),
    "negative_divide_zero_is_neg_inf": bool(np.isneginf(minus_inf)),
    "zero_divide_zero_is_nan": bool(np.isnan(nan_value)),
    "nan_greater_than_zero": bool(nan_value > 0.0),
    "signed_zero_signbit": bool(np.signbit(signed_zero)),
    "harmonic_zero_result": float(harmonic_parallel),
}
assert CHECKS["floating_point"]["positive_divide_zero_is_pos_inf"]
assert CHECKS["floating_point"]["zero_divide_zero_is_nan"]
assert CHECKS["floating_point"]["nan_greater_than_zero"] is False
assert CHECKS["floating_point"]["signed_zero_signbit"]
fig_caption(float_path)


## Efficiency: locality before cleverness

The chapter's efficiency advice is conservative: first write the direct version, then measure, and expect memory access to matter. The plot below makes that advice concrete without depending on a specific CPU benchmark. The left panel counts cache-line switches for two traversals over the same row-major image. Both visit the same number of pixels and do the same amount of arithmetic, but the column-major path jumps between memory lines much more often.

The right panel sketches level of detail. Farther objects can often use fewer triangles because their screen footprint is smaller. The point is not that one curve is universal; the point is that performance choices should be tied to measured bottlenecks and visible error.


In [ ]:
def cache_line_switches(width, height, traversal, pixels_per_line=8):
    if traversal == "row-major":
        coords = [(y, x) for y in range(height) for x in range(width)]
    elif traversal == "column-major":
        coords = [(y, x) for x in range(width) for y in range(height)]
    else:
        raise ValueError(traversal)
    lines = np.array([(y * width + x) // pixels_per_line for y, x in coords])
    return int(1 + np.count_nonzero(lines[1:] != lines[:-1]))

mem_width, mem_height = 64, 48
row_switches = cache_line_switches(mem_width, mem_height, "row-major")
col_switches = cache_line_switches(mem_width, mem_height, "column-major")

small_w, small_h = 16, 10
row_order = np.array([y * small_w + x for y in range(small_h) for x in range(small_w)]).reshape(small_h, small_w)
col_order = np.empty((small_h, small_w), dtype=int)
for k, (y, x) in enumerate((y, x) for x in range(small_w) for y in range(small_h)):
    col_order[y, x] = k

lod_distance = np.array([1.0, 2.0, 4.0, 8.0])
lod_triangles = np.array([24000, 7200, 2100, 620])
frame_ms = 1.6 + lod_triangles / 9500.0
screen_error_px = np.array([0.35, 0.55, 0.85, 1.35])

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), gridspec_kw={"width_ratios": [1.0, 1.0, 1.2]})
axes[0].imshow(row_order, cmap="viridis")
axes[0].set_title(f"row-major\n{row_switches} cache-line visits")
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[1].imshow(col_order, cmap="viridis")
axes[1].set_title(f"column-major\n{col_switches} cache-line visits")
axes[1].set_xticks([])
axes[1].set_yticks([])
ax2 = axes[2]
ax2.plot(lod_distance, frame_ms, "o-", color=PALETTE["blue"], label="estimated frame ms")
ax2.set_xlabel("relative viewing distance")
ax2.set_ylabel("frame cost (ms)", color=PALETTE["blue"])
ax2.tick_params(axis="y", labelcolor=PALETTE["blue"])
ax2.grid(True, color="#d7dde5", alpha=0.85)
ax3 = ax2.twinx()
ax3.plot(lod_distance, screen_error_px, "s--", color=PALETTE["red"], label="screen error")
ax3.set_ylabel("screen error (pixels)", color=PALETTE["red"])
ax3.tick_params(axis="y", labelcolor=PALETTE["red"])
ax2.set_title("LOD: fewer triangles when error stays small")
for d, tri in zip(lod_distance, lod_triangles):
    ax2.text(d, 1.6 + tri / 9500.0 + 0.08, f"{tri//1000}k", ha="center", fontsize=8, color=PALETTE["ink"])
fig.suptitle("Efficiency depends on data movement and visible detail", fontsize=12, color=PALETTE["ink"])

eff_path = remember(save_matplotlib(fig, TOPIC, "efficiency-locality-lod-budget.png"), "figures")
close(fig)
CHECKS["efficiency"] = {
    "row_major_cache_line_visits": row_switches,
    "column_major_cache_line_visits": col_switches,
    "column_to_row_visit_ratio": float(col_switches / row_switches),
    "lod_triangles_monotone_decreasing": bool(np.all(np.diff(lod_triangles) < 0)),
    "frame_ms_monotone_decreasing": bool(np.all(np.diff(frame_ms) < 0)),
}
assert row_switches < col_switches
assert CHECKS["efficiency"]["lod_triangles_monotone_decreasing"]
fig_caption(eff_path)


## Visual debugging

The chapter's debugging advice is unusually practical: do not rely only on stepping through a program that runs the same code millions of times. Make an image of the internal state. A normal map can show whether surface directions are plausible. An object-ID map can show whether visibility is selecting the intended primitive. A work heatmap can show where the renderer spends time. A red mask can make invalid ranges impossible to miss.

The artifact below is a synthetic debug frame. It is not meant to be pretty. It is meant to answer questions quickly: are normals normalized, which object won visibility, where is the expensive region, and where did values leave their expected range?


In [ ]:
debug_size = 220
y, x = np.mgrid[-1:1:complex(debug_size), -1:1:complex(debug_size)]
r2 = x * x + y * y
sphere_mask = r2 <= 1.0
z = np.sqrt(np.clip(1.0 - r2, 0.0, 1.0))
normals = np.dstack([x, y, z])
normal_rgb = np.ones((debug_size, debug_size, 3)) * 0.96
normal_rgb[sphere_mask] = 0.5 * (normals[sphere_mask] + 1.0)

object_id = np.zeros((debug_size, debug_size), dtype=int)
object_id[sphere_mask & (x < -0.18)] = 1
object_id[sphere_mask & (np.abs(x) <= 0.18)] = 2
object_id[sphere_mask & (x > 0.18)] = 3
id_palette = np.array([
    [0.96, 0.97, 0.98],
    [0.20, 0.45, 0.78],
    [0.88, 0.58, 0.18],
    [0.40, 0.68, 0.42],
])
id_rgb = id_palette[object_id]

work = np.zeros((debug_size, debug_size), dtype=float)
work[sphere_mask] = 8 + 22 * np.exp(-((x[sphere_mask] - 0.35) ** 2 + (y[sphere_mask] + 0.2) ** 2) / 0.12) + 10 * (z[sphere_mask] < 0.45)
work_norm = np.clip(work / max(work.max(), 1.0), 0, 1)
work_rgb = plt.get_cmap("magma")(work_norm)[..., :3]
work_rgb[~sphere_mask] = np.array([0.96, 0.97, 0.98])

buggy_normals = normals.copy()
buggy_normals[..., 0] *= 1.28
buggy_length = np.linalg.norm(buggy_normals, axis=-1)
invalid_before = sphere_mask & (np.abs(buggy_length - 1.0) > 0.08)
fixed_normals = buggy_normals.copy()
fixed_normals[sphere_mask] = fixed_normals[sphere_mask] / buggy_length[sphere_mask, None]
fixed_length = np.linalg.norm(fixed_normals, axis=-1)
invalid_after = sphere_mask & (np.abs(fixed_length - 1.0) > 0.08)
mask_rgb = np.ones((debug_size, debug_size, 3)) * np.array([0.96, 0.97, 0.98])
mask_rgb[sphere_mask] = np.array([0.78, 0.84, 0.91])
mask_rgb[invalid_before] = np.array([0.92, 0.05, 0.08])

fig, axes = plt.subplots(1, 4, figsize=(12, 3.6))
for ax, img, title in zip(
    axes,
    [normal_rgb, id_rgb, work_rgb, mask_rgb],
    ["normal -> RGB", "object IDs", "work heatmap", "out-of-range mask"],
):
    ax.imshow(img)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
fig.suptitle("Images as coded debugging output", fontsize=12, color=PALETTE["ink"])
debug_path = remember(save_matplotlib(fig, TOPIC, "visual-debugging-channels.png"), "figures")
close(fig)

hypothesis_rows = [
    {"hypothesis": "normal x component scaled before shading", "test_image": "normal length error mask", "invalid_pixels": int(invalid_before.sum()), "after_local_fix": int(invalid_after.sum())},
    {"hypothesis": "object visibility choosing wrong primitive", "test_image": "object ID colors", "invalid_pixels": int(np.count_nonzero(object_id == 0)), "after_local_fix": int(np.count_nonzero(object_id == 0))},
    {"hypothesis": "expensive rays cluster near silhouette", "test_image": "work heatmap", "invalid_pixels": int(np.count_nonzero(work > 28)), "after_local_fix": int(np.count_nonzero(work > 28))},
]
hypothesis_path = remember(save_table_csv(hypothesis_rows, TOPIC, "debugging-hypothesis-tests.csv"), "tables")
CHECKS["visual_debugging"] = {
    "invalid_pixels_before_fix": int(invalid_before.sum()),
    "invalid_pixels_after_fix": int(invalid_after.sum()),
    "normal_length_max_error_after_fix": float(np.max(np.abs(fixed_length[sphere_mask] - 1.0))),
    "object_id_count": int(len(np.unique(object_id[sphere_mask]))),
    "max_work_value": float(work.max()),
}
assert CHECKS["visual_debugging"]["invalid_pixels_after_fix"] < CHECKS["visual_debugging"]["invalid_pixels_before_fix"]
assert CHECKS["visual_debugging"]["object_id_count"] == 3
fig_caption(debug_path)
display_artifact(hypothesis_path)


## Applied lab: debug a small rendering failure

Use the generated debug targets as a disciplined experiment, not as decoration.

1. Start from the symptom: the red mask says a group of pixels has a normal length outside the expected range.
2. Form a hypothesis: one component of the normal was scaled during a coordinate conversion, but the vector was not renormalized.
3. Test locally: renormalize only those surface samples and compare the invalid-pixel count.
4. Keep the useful output: the normal map and mask are worth leaving in the renderer as optional debug views because they turn a conceptual error into a visible pattern.

The table saved above is the lab record. It is small, but it mirrors the chapter's advice: make the program reveal its internal state, then test one explanation at a time.


In [ ]:
lab_report = {
    "symptom": "debug mask marks normals whose length is outside tolerance",
    "hypothesis": "a coordinate conversion scaled one normal component without renormalizing",
    "invalid_pixels_before_fix": CHECKS["visual_debugging"]["invalid_pixels_before_fix"],
    "invalid_pixels_after_fix": CHECKS["visual_debugging"]["invalid_pixels_after_fix"],
    "hypothesis_supported": CHECKS["visual_debugging"]["invalid_pixels_after_fix"] < CHECKS["visual_debugging"]["invalid_pixels_before_fix"],
}
lab_path = remember(save_json(lab_report, TOPIC, "visual-debugging-lab-report.json"), "checks")
display_artifact(lab_path)
lab_report


## Sanity checks

The final cell checks both file integrity and chapter-specific invariants. These are deliberately more concrete than "the notebook ran." They assert that the application graph is not a single-use taxonomy, the z-buffer really chooses the nearer sample, ray intersections are finite, homogeneous projection has valid `w`, IEEE sentinel behavior matches the branch logic used in the prose, coherent memory traversal has fewer cache-line visits, and the visual-debugging fix reduces the invalid mask.


In [ ]:
invariant_path = remember(save_json(CHECKS, TOPIC, "chapter-01-invariants.json"), "checks")

all_paths = []
for kind in ["figures", "html", "tables", "checks"]:
    all_paths.extend(ARTIFACTS[kind])

final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "source_span": {"printed_pages": PRINTED_PAGES, "physical_pdf_pages": PDF_PAGES},
    "artifact_counts": {kind: len(paths) for kind, paths in ARTIFACTS.items()},
    "figures": [book_relative(path) for path in ARTIFACTS["figures"]],
    "html": [book_relative(path) for path in ARTIFACTS["html"]],
    "tables": [book_relative(path) for path in ARTIFACTS["tables"]],
    "checks": CHECKS,
}
final_path = remember(save_json(final_report, TOPIC, "final-sanity.json"), "checks")
final_report["artifact_counts"] = {kind: len(paths) for kind, paths in ARTIFACTS.items()}
save_json(final_report, TOPIC, "final-sanity.json")
all_paths.append(final_path)

artifact_records = assert_artifacts(all_paths)
image_records = [assert_nonblank_image(path) for path in ARTIFACTS["figures"]]

assert CHECKS["application_graph"]["connected"]
assert CHECKS["pipeline_responsibility"]["pipeline_stage_count"] == 6
assert CHECKS["rendering_routes"]["z_buffer_closer_wins"]
assert CHECKS["rendering_routes"]["ray_hit_pixels"] > 0
assert CHECKS["homogeneous_pipeline"]["clip_w_all_positive"]
assert CHECKS["homogeneous_pipeline"]["ndc_all_finite"]
assert CHECKS["floating_point"]["nan_greater_than_zero"] is False
assert CHECKS["efficiency"]["row_major_cache_line_visits"] < CHECKS["efficiency"]["column_major_cache_line_visits"]
assert CHECKS["visual_debugging"]["invalid_pixels_after_fix"] < CHECKS["visual_debugging"]["invalid_pixels_before_fix"]

summary = {
    "artifact_count": len(artifact_records),
    "nonblank_figure_count": len(image_records),
    "final_report": book_relative(final_path),
    "invariant_report": book_relative(invariant_path),
}
display_artifact(final_path)
summary


## Takeaways

Chapter 01 is an orientation chapter, but it already contains the operating habits of graphics work.

Graphics projects combine areas. Modeling, rendering, animation, image processing, visualization, interaction, scanning, and computational photography overlap in real applications, so a useful codebase exposes data in ways that can be shared and inspected.

APIs are contracts, not explanations. They hide device details, but the pipeline still has concrete intermediate states: vertices, 4-vectors, clip coordinates, coverage masks, depth values, colors, and images.

Rasterization and ray tracing are two visibility routes. Rasterization asks which samples a primitive covers and resolves visibility with a depth buffer. Ray tracing asks which object each ray hits first. Later chapters develop both routes in detail.

Floating-point edge cases are design tools when used intentionally. Infinities, NaNs, and signed zeros can simplify code paths, but they should be paired with assertions and debug views when they mark impossible states.

Efficiency is empirical. A straightforward implementation plus profiling is a better starting point than clever code that fights memory locality or spends triangles where they have no visible effect.

The most graphics-specific debugging tool is the image itself. If a variable exists per pixel, per ray, or per primitive, it can often be turned into a color-coded output that makes the bug visible.
